In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Employee RDD Analysis") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/26 16:17:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:

sc = spark.sparkContext

26/06/26 16:17:20 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [3]:

rdd = sc.textFile("data/employees.csv")

In [4]:
# Removing header
header = rdd.first()
employee_rdd = rdd.filter(lambda x: x != header)

In [5]:

# Convert records into tuples
employee_rdd = employee_rdd.map(
    lambda x: x.split(",")
)

In [6]:
# ALL EMPLOYEES SORTED BY SALARY DESCENDING
sorted_employees = employee_rdd.sortBy(
    lambda x: int(x[3]),
    ascending=False
)

for emp in sorted_employees.collect():
    print(emp)

[Stage 3:>                                                          (0 + 2) / 2]

['4', 'Priya', 'Finance', '70000']
['3', 'Neha', 'IT', '65000']
['7', 'Rohit', 'Finance', '60000']
['1', 'Amit', 'IT', '55000']
['5', 'Karan', 'IT', '50000']
['6', 'Simran', 'HR', '45000']
['2', 'Rahul', 'HR', '40000']


In [7]:

# Department Wise Salary Totals
department_salary = (
    employee_rdd
    .map(lambda x: (x[2], int(x[3])))
    .reduceByKey(lambda a, b: a + b)
)

for dept, total in department_salary.collect():
    print(f"{dept}: {total}")

IT: 170000
HR: 85000
Finance: 130000


In [8]:
# TOP 3 HIGHEST PAID EMPLOYEES
top3 = sorted_employees.take(3)
for emp in top3:
    print(emp)

# Save Top 3 Employees to File
top3_rdd = sc.parallelize(
    [",".join(emp) for emp in top3]
)
top3_rdd.saveAsTextFile(
    "output/top3_employees"
)

print("\nTop 3 employees saved successfully.")

['4', 'Priya', 'Finance', '70000']
['3', 'Neha', 'IT', '65000']
['7', 'Rohit', 'Finance', '60000']



Top 3 employees saved successfully.


In [9]:
spark.stop()